In [2]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import random
from pathlib import Path
import os 
import sys
from collections import Counter

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
BASE_DIR = Path.cwd() 
print(BASE_DIR)

c:\Users\Neezy\Videos\attacks_improved_v2\Attacks-Improved_v2\decision_tree


In [5]:
project_root = os.path.abspath('..')
if project_root not in sys.path: 
    sys.path.append(project_root)

In [6]:
#### log initialization
from utils.logs import logger_module 
logger = logger_module(logger_name=__name__) 

In [7]:
# import attacks
from attack_functionality.general_attack import GeneralAttack 
attack = GeneralAttack()

In [8]:
data = pd.read_csv(BASE_DIR.parent/'data/ds_with_sample_name.csv')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [9]:
X = data.iloc[:, 2:-1].values
y = data.iloc[:, -1].values
original_indices = np.arange(len(X)) 
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(X, y, original_indices, test_size=0.2, random_state=42)
classifier = DecisionTreeClassifier(random_state=42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:\n {cm}") 
cr = classification_report(y_test, y_pred)
print(f"Classification report:\n {cr}")

Accuracy: 97.43%
Confusion Matrix:
 [[105  12]
 [  3 463]]
Classification report:
               precision    recall  f1-score   support

           0       0.97      0.90      0.93       117
           1       0.97      0.99      0.98       466

    accuracy                           0.97       583
   macro avg       0.97      0.95      0.96       583
weighted avg       0.97      0.97      0.97       583



In [10]:
importances = classifier.feature_importances_
feature_imp_df = pd.DataFrame({'Feature': data.iloc[:, 2:-1].columns, 'Gini Importance': importances}).sort_values(
    'Gini Importance', ascending=False)

important_features = []
importances_to_numpy = feature_imp_df.to_numpy()
for i in range(0,20):
     important_features.append(importances_to_numpy[i][0])

In [11]:
important_features.append('label')

In [12]:
df = data[important_features]

In [13]:
df.head()

,RegEnumKeyExW,RegQueryValueExW,LdrLoadDll,NtQueryDirectoryFile,CoUninitialize,NtDeleteKey,NtAllocateVirtualMemory,NtTerminateProcess,CertControlStore,DrawTextExW,GetSystemMetrics,NtFreeVirtualMemory,NtQuerySystemInformation,NtEnumerateKey,NtCreateFile,NtClose,LdrUnloadDll,CreateDirectoryW,RegCloseKey,RegQueryInfoKeyW,label
0,128,1308,253,920,40,0,3346,75,0,86,670,3771,7,0,7398,18734,74,9,7834,72,1
1,0,0,2,0,0,0,6,3,0,0,0,1,0,0,0,3,0,0,0,0,1
2,38,317,70,338,0,0,829,3,0,0,6,291,4,0,1213,1977,7,3,136,12,1
3,21,182,32,17,0,0,610,0,0,0,1,374,2,0,95,134,2,4,66,6,1
4,0,0,3,5,1,0,6,3,0,0,0,3,0,0,2,51,2,0,1,0,1


In [14]:
X_imp_feat = df.iloc[:, :-1].values
y_imp_feat = df.iloc[:, -1].values
original_indices = np.arange(len(X_imp_feat)) 
X_train_fi, X_test_fi, y_train_fi, y_test_fi, idx_train, idx_test = train_test_split(X_imp_feat, y_imp_feat, original_indices, test_size=0.2, random_state=42)
classifier_dt_imp = DecisionTreeClassifier(random_state=42)
classifier_dt_imp.fit(X_train_fi, y_train_fi)
y_pred_imp = classifier_dt_imp.predict(X_test_fi)
accuracy_fi = accuracy_score(y_test_fi, y_pred_imp)
print(f'Accuracy: {accuracy_fi * 100:.2f}%')
cm_fi = confusion_matrix(y_test_fi, y_pred_imp)
print(f"Confusion Matrix:\n {cm_fi}") 
cr_fi = classification_report(y_test_fi, y_pred_imp)
print(f"Classification report:\n {cr_fi}")

Accuracy: 97.43%
Confusion Matrix:
 [[105  12]
 [  3 463]]
Classification report:
               precision    recall  f1-score   support

           0       0.97      0.90      0.93       117
           1       0.97      0.99      0.98       466

    accuracy                           0.97       583
   macro avg       0.97      0.95      0.96       583
weighted avg       0.97      0.97      0.97       583



In [15]:
ransomware_training_sample_indices = np.where(y_train==1)
idx = np.random.choice(ransomware_training_sample_indices[0], 580, replace=False)
xtrain_poison_candidates = X_train_fi[idx] 

In [16]:
print(len(xtrain_poison_candidates))

580


In [17]:
perturbation=20
asamples, adv_idx = attack.generate_adversarial_sample(idx, xtrain_poison_candidates, perturbation, classifier_dt_imp)

The average  l2 distance 1.1257541875914818 across 575 samples
The biggest  l2 distance 14.0 across 575 samples
The smallest l2 distance 1.0 across 575 samples


In [18]:
print(len(asamples))
print(f"AGR: {len(asamples)/len(xtrain_poison_candidates):.4f}")

575
AGR: 0.9914


In [19]:
#np.save("dt_adversarial_samples_top_100.npy", np.array(asamples))

In [20]:
### Adversarial Retrain ###  
mask = np.ones(X_train_fi.shape[0], dtype=bool)
mask[adv_idx] = False

X_train_fi = X_train_fi[mask]
y_train_fi = y_train_fi[mask]

print(f"length of Xtrain with successful base adversarial samples removed: {len(X_train_fi)}")

### Select a hold out test set of the generated adversarial samples
k = round(0.2 * len(asamples)) 
print(f"20% of the adversarial samples held out for testing: {k}")
remove_samples_from_asamples = random.sample(asamples, k)
print(len(remove_samples_from_asamples))

### remove test set from total adversarial samples and add adversarial samples to the training set 
result = [item for item in asamples if item not in remove_samples_from_asamples]
np_result = np.array(result)
print(len(result))
poisoned_xtrain = np.concatenate((X_train_fi, np_result))
print(f"Total length of poisoned Xtrain: {len(poisoned_xtrain)}")

### Add the corresponding amount of ytrain adversarial samples. (correctly labelled as ransomware) 
adv_y = []
for i in range(len(np_result)): 
    adv_y.append(1) 
    
print(len(adv_y))

np_advy = np.array(adv_y)
poisoned_ytrain = np.concatenate((y_train_fi, np_advy))
print(f"Total length of poisoned Ytrain: {len(poisoned_ytrain)}") 

if len(poisoned_ytrain) == len(poisoned_xtrain): 
    print("Poisoned Xtrain and ytrain are aligned, posioned training can now commence")
else: 
    print("Not aligned, something is not aligned, check k value")

length of Xtrain with successful base adversarial samples removed: 1755
20% of the adversarial samples held out for testing: 115
115
324
Total length of poisoned Xtrain: 2079
324
Total length of poisoned Ytrain: 2079
Poisoned Xtrain and ytrain are aligned, posioned training can now commence


In [21]:
poisoned_classifier = DecisionTreeClassifier(random_state=42)
poisoned_classifier.fit(poisoned_xtrain, poisoned_ytrain)
y_pred_poisoned = poisoned_classifier.predict(X_test_fi)
poisoned_accuracy = accuracy_score(y_test_fi, y_pred_poisoned)
print(f'Accuracy trained on poisoned samples: {poisoned_accuracy * 100:.2f}%')
cm_poisoned = confusion_matrix(y_test_fi, y_pred_poisoned)
print(f"Confusion Matrix trained on poisoned samples:\n {cm_poisoned}") 
cr_poisoned = classification_report(y_test_fi, y_pred_poisoned)
print(f"Classification report trained on poisoned samples:\n {cr_poisoned}")

Accuracy trained on poisoned samples: 96.74%
Confusion Matrix trained on poisoned samples:
 [[104  13]
 [  6 460]]
Classification report trained on poisoned samples:
               precision    recall  f1-score   support

           0       0.95      0.89      0.92       117
           1       0.97      0.99      0.98       466

    accuracy                           0.97       583
   macro avg       0.96      0.94      0.95       583
weighted avg       0.97      0.97      0.97       583



In [22]:
def make_pred_poisoned(samples):
    predictions = [] 
    for sample in samples:
        single_pred = poisoned_classifier.predict([sample])
        predictions.append(single_pred)
        print(single_pred) 
        if single_pred[0] == 1: 
            print("Correctly identified adversarial ransomware sample") 
        else: 
            print("Adversarial sample still adversarial")

    zero_counter = 0 
    one_counter = 0 
    for pred in predictions:
        if pred != 1: 
            zero_counter +=1 
        else:
            one_counter+=1 
        
    
    print(f"Samples still adversarial: {zero_counter}")
    print(f"adversarial_samples identified: {one_counter}")

make_pred_poisoned(remove_samples_from_asamples)

[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correc

In [23]:
###Model transfer ###
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [24]:
### Target Models ####
##  Trained on original data
#   No adversarial samples are added to the training data 

### Random Forest ####
classifier_rf = RandomForestClassifier(n_estimators=100, random_state=42)
classifier_rf.fit(X_train_fi, y_train_fi)
y_pred_rf = classifier_rf.predict(X_test_fi)
accuracy_rf = accuracy_score(y_test_fi, y_pred_rf) 
print(f"Accuracy of Random Forest Target Model: {accuracy_score(y_test, y_pred):.2f}")

### XGboost #### 
params = {
    'objective':'binary:logistic',
    'max_depth':4,
    'learning_rate':0.1,
    'n_estimators':100,
    'alpha':10
}

model = XGBClassifier(**params)
model.fit(X_train_fi, y_train_fi)
y_pred_xgb = model.predict(X_test_fi)
accuracy_xgb = accuracy_score(y_test_fi, y_pred_xgb)
print(f'Accuracy of the XGboost Model: {accuracy_xgb * 100:.2f}%') 

### SVM Classifier Kernel=Poly ###
svm_classifier = SVC(kernel='poly', C=1.0, random_state=42)
svm_classifier.fit(X_train_fi, y_train_fi)
y_pred = svm_classifier.predict(X_test_fi)
print(f"Accuracy of the SVM Classifier (Kernel=Poly): {accuracy_score(y_test, y_pred):.2f}")

# ### SVM Classifier Kernel=Linear ###
# svm_classifier_linear = SVC(kernel='linear', C=4.0, random_state=42)
# svm_classifier_linear.fit(X_train_fi, y_train_fi)
# y_pred = svm_classifier_linear.predict(X_test_fi)
# print(f"Accuracy of the SVM Classifier (Kernel=Linear) : {accuracy_score(y_test, y_pred):.2f}")

### Logistic Regression ###
LR_model = LogisticRegression(solver='saga',max_iter=5000)
LR_model.fit(X_train_fi, y_train_fi)
ypred_LR = LR_model.predict(X_test_fi)
accuracy = accuracy_score(y_test_fi, ypred_LR)
print(f"accuracy of the Logistic Regression Model: {round(accuracy,2)}")

Accuracy of Random Forest Target Model: 0.97
Accuracy of the XGboost Model: 97.43%
Accuracy of the SVM Classifier (Kernel=Poly): 0.80
accuracy of the Logistic Regression Model: 0.95


c:\Users\Neezy\anaconda3\envs\tf\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [25]:
def evasion_attack(adversarial_samples,
                   RF_model,
                   XGB_Model, 
                   SVM_poly_model, 
                   Logistic_reg_model):
    
    predictions = {
        "Random_forest":[],
        "XGB":[],
        "SVM_poly": [], 
        "Logistic_Regression":[]   
    }
    
    models = {
        "Random_forest": RF_model,
        "XGB":XGB_Model,
        "SVM_poly": SVM_poly_model,  
        "Logistic_Regression":Logistic_reg_model
    }
    
    for name, model in models.items():
        preds = model.predict(adversarial_samples)
        predictions[name] = preds.tolist()
            
    
    return predictions

In [26]:
res = evasion_attack(asamples, classifier_rf, model, svm_classifier, LR_model)
evaded_ids=[]
evaded_samples = []
original_positions = []
for model_name, preds in res.items():
    print(f"\n {model_name}: predictions")
    total = len(preds)
    count_1 = sum(1 for p in preds if p==1)
    count_2 = total-count_1 
    
    model_evaded_positions = []
    model_evaded_samples = []
    
    for i, p in enumerate(preds): 
        if p == 0: 
           xtrain_position = adv_idx[i] 
           original_position = idx_train[xtrain_position]
           model_evaded_positions.append(original_position)
            
    if model_evaded_positions:
        for og_position in model_evaded_positions:
            og_sample = data.iloc[og_position]['Sample Name']
            model_evaded_samples.append(og_sample)
        evaded_samples.extend(model_evaded_samples)
        original_positions.extend(model_evaded_positions)
    else: 
        print(f"\n No samples evaded for {model_name}")
    
    print(f"\n {model_name} evasion rate: {count_2}/{total} samples evaded")
    
    sample_names = [s.rsplit('_', 1)[0] for s in model_evaded_samples]
    print("\n evaded samples: ", Counter(sample_names))      


 Random_forest: predictions

 Random_forest evasion rate: 1/575 samples evaded

 evaded samples:  Counter({'TeslaCrypt': 1})

 XGB: predictions

 XGB evasion rate: 2/575 samples evaded

 evaded samples:  Counter({'trickbot_Sample': 1, 'None': 1})

 SVM_poly: predictions

 No samples evaded for SVM_poly

 SVM_poly evasion rate: 0/575 samples evaded

 evaded samples:  Counter()

 Logistic_Regression: predictions

 Logistic_Regression evasion rate: 26/575 samples evaded

 evaded samples:  Counter({'Babuk': 6, 'Magniber': 6, 'Stop': 3, 'None': 2, 'HelloKitty': 1, 'FuxSocy': 1, 'TeslaCrypt': 1, 'Jackpot': 1, 'BlackBasta': 1, 'VHDLocker': 1, 'Chaos': 1, 'WannaCry': 1, 'Spora': 1})
